In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pickle
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

DATA_PATH = '/content/drive/MyDrive/GAMEEMO_processed/gameemo_preprocessed.pkl'
SAVE_DIR = '/content/drive/MyDrive/GAMEEMO_processed/results'
os.makedirs(SAVE_DIR, exist_ok=True)

with open(DATA_PATH, 'rb') as f:
    data = pickle.load(f)

X = data['X']
y = data['y']
subject_ids = data['subject_ids']
n_classes = len(data['label_names'])

print('X shape:', X.shape)
print('y shape:', y.shape)
print('classes:', data['label_names'])

X shape: (8288, 512, 14)
y shape: (8288,)
classes: ['boring', 'calm', 'horror', 'funny']


In [ ]:
subject_ids[0]

np.int64(26)

In [ ]:
# we did not save a separate trial id during preprocessing, but we can reconstruct it
# windows are appended in file order, and within a file every window shares the same subject id and label
# so a combination of subject id and label uniquely identifies the trial each window came from

game_labels = data['label_names']
trial_ids = np.array([f'{s}_{l}' for s, l in zip(subject_ids, y)])

n_unique_trials = len(np.unique(trial_ids))
print('total trials represented:', n_unique_trials, 'expected around', 28 * 4)

total trials represented: 112 expected around 112


In [ ]:
def get_subject_dependent_split(X, y, trial_ids, test_size=0.2, seed=42):
    # subject dependent here means: every subject can appear in both train and test,
    # but a given trial (subject + game recording) only ever appears on one side
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups=trial_ids))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


def get_cross_subject_split(X, y, subject_ids, test_size=0.2, seed=42):
    # cross subject: split by subject id directly, so test subjects are completely unseen during training
    splitter = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    train_idx, test_idx = next(splitter.split(X, y, groups=subject_ids))
    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]

In [ ]:
def prepare_for_eegnet(X):
    X_t = np.transpose(X, (0, 2, 1))
    X_t = X_t[..., np.newaxis]
    return X_t


def build_eegnet(n_channels, n_timepoints, n_classes, dropout_rate=0.5):
    inputs = layers.Input(shape=(n_channels, n_timepoints, 1))

    x = layers.Conv2D(8, (1, 64), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)

    x = layers.DepthwiseConv2D((n_channels, 1), use_bias=False, depth_multiplier=2,
                                depthwise_constraint=tf.keras.constraints.max_norm(1.))(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 4))(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.SeparableConv2D(16, (1, 16), use_bias=False, padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('elu')(x)
    x = layers.AveragePooling2D((1, 8))(x)
    x = layers.Dropout(dropout_rate)(x)

    x = layers.Flatten()(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
def build_1d_cnn(n_timepoints, n_channels, n_classes):
    inputs = layers.Input(shape=(n_timepoints, n_channels))

    x = layers.Conv1D(32, kernel_size=7, activation='relu', padding='same')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(64, kernel_size=5, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)

    x = layers.Conv1D(128, kernel_size=3, activation='relu', padding='same')(x)
    x = layers.BatchNormalization()(x)
    x = layers.GlobalAveragePooling1D()(x)

    x = layers.Dense(64, activation='relu')(x)
    x = layers.Dropout(0.4)(x)
    outputs = layers.Dense(n_classes, activation='softmax')(x)

    model = models.Model(inputs, outputs)
    return model

In [ ]:
def train_and_evaluate(model, X_train, y_train, X_test, y_test, model_name, protocol_name, epochs=50, batch_size=32):
    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    checkpoint_path = os.path.join(SAVE_DIR, f'{model_name}_{protocol_name}_best.keras')

    callback_list = [
        callbacks.EarlyStopping(patience=10, restore_best_weights=True),
        callbacks.ModelCheckpoint(checkpoint_path, save_best_only=True, monitor='val_accuracy'),
        callbacks.ReduceLROnPlateau(factor=0.5, patience=5)
    ]

    history = model.fit(
        X_train, y_train,
        validation_split=0.15,
        epochs=epochs,
        batch_size=batch_size,
        callbacks=callback_list,
        verbose=1
    )

    y_pred = np.argmax(model.predict(X_test), axis=1)

    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted')

    print(f'{model_name} ({protocol_name}) test accuracy: {acc:.4f}, f1: {f1:.4f}')
    print(classification_report(y_test, y_pred, target_names=data['label_names']))

    result = {
        'model': model_name,
        'protocol': protocol_name,
        'accuracy': acc,
        'f1_score': f1,
        'confusion_matrix': confusion_matrix(y_test, y_pred).tolist(),
        'history': history.history
    }

    return result, model

# **EEGNet Subject Dependent (Trial level split)**

In [ ]:
X_train, X_test, y_train, y_test = get_subject_dependent_split(X, y, trial_ids)

X_train_eeg = prepare_for_eegnet(X_train)
X_test_eeg = prepare_for_eegnet(X_test)

eegnet_model = build_eegnet(n_channels=14, n_timepoints=512, n_classes=n_classes)
result_eegnet_sd, eegnet_model_sd = train_and_evaluate(
    eegnet_model, X_train_eeg, y_train, X_test_eeg, y_test,
    model_name='eegnet', protocol_name='subject_dependent'
)

Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 21s 66ms/step - accuracy: 0.2588 - loss: 1.4574 - val_accuracy: 0.2611 - val_loss: 1.3854 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3030 - loss: 1.3837 - val_accuracy: 0.2702 - val_loss: 1.3939 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3537 - loss: 1.3128 - val_accuracy: 0.3219 - val_loss: 1.4424 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3959 - loss: 1.2490 - val_accuracy: 0.2723 - val_loss: 1.3969 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4573 - loss: 1.1619 - val_accuracy: 0.2328 - val_loss: 1.5126 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4980 - loss: 1.1008 - val_accuracy: 0.2905 - val_loss: 1.4240 - learning_rate: 0.0010
Epoch 7/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5389 - loss: 1.0390 

# EEGNet Cross Subject

In [ ]:
X_train_cs, X_test_cs, y_train_cs, y_test_cs = get_cross_subject_split(X, y, subject_ids)

X_train_cs_eeg = prepare_for_eegnet(X_train_cs)
X_test_cs_eeg = prepare_for_eegnet(X_test_cs)

eegnet_model_2 = build_eegnet(n_channels=14, n_timepoints=512, n_classes=n_classes)
result_eegnet_cs, eegnet_model_cs = train_and_evaluate(
    eegnet_model_2, X_train_cs_eeg, y_train_cs, X_test_cs_eeg, y_test_cs,
    model_name='eegnet', protocol_name='cross_subject'
)

Epoch 1/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step - accuracy: 0.2528 - loss: 1.4527 - val_accuracy: 0.3019 - val_loss: 1.3817 - learning_rate: 0.0010
Epoch 2/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.2761 - loss: 1.3953 - val_accuracy: 0.3265 - val_loss: 1.3515 - learning_rate: 0.0010
Epoch 3/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.3254 - loss: 1.3409 - val_accuracy: 0.3040 - val_loss: 1.3232 - learning_rate: 0.0010
Epoch 4/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3875 - loss: 1.2625 - val_accuracy: 0.4217 - val_loss: 1.2518 - learning_rate: 0.0010
Epoch 5/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.4589 - loss: 1.1678 - val_accuracy: 0.3941 - val_loss: 1.1879 - learning_rate: 0.0010
Epoch 6/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5006 - loss: 1.1074 - val_accuracy: 0.3859 - val_loss: 1.2166 - learning_rate: 0.0010
Epoch 7/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5171 - loss: 1.0591 

# 1D-CNN Subject Dependent (Trial Level Split)

In [ ]:
cnn_model = build_1d_cnn(n_timepoints=512, n_channels=14, n_classes=n_classes)
result_cnn_sd, cnn_model_sd = train_and_evaluate(
    cnn_model, X_train, y_train, X_test, y_test,
    model_name='cnn1d', protocol_name='subject_dependent'
)

Epoch 1/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 12s 36ms/step - accuracy: 0.5339 - loss: 1.0721 - val_accuracy: 0.4312 - val_loss: 1.2371 - learning_rate: 0.0010
Epoch 2/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.7394 - loss: 0.6799 - val_accuracy: 0.6073 - val_loss: 1.1240 - learning_rate: 0.0010
Epoch 3/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8189 - loss: 0.4962 - val_accuracy: 0.6538 - val_loss: 1.1894 - learning_rate: 0.0010
Epoch 4/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.8542 - loss: 0.4007 - val_accuracy: 0.6032 - val_loss: 1.0987 - learning_rate: 0.0010
Epoch 5/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 8ms/step - accuracy: 0.8846 - loss: 0.3229 - val_accuracy: 0.6599 - val_loss: 1.2774 - learning_rate: 0.0010
Epoch 6/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.9005 - loss: 0.2704 - val_accuracy: 0.7308 - val_loss: 1.0600 - learning_rate: 0.0010
Epoch 7/50
175/175 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9180 - loss: 0.2373 

# 1D-CNN Cross Subject

In [ ]:
cnn_model_2 = build_1d_cnn(n_timepoints=512, n_channels=14, n_classes=n_classes)
result_cnn_cs, cnn_model_cs = train_and_evaluate(
    cnn_model_2, X_train_cs, y_train_cs, X_test_cs, y_test_cs,
    model_name='cnn1d', protocol_name='cross_subject'
)

Epoch 1/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 13s 46ms/step - accuracy: 0.5286 - loss: 1.0671 - val_accuracy: 0.3623 - val_loss: 1.1880 - learning_rate: 0.0010
Epoch 2/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7187 - loss: 0.7051 - val_accuracy: 0.6530 - val_loss: 0.8790 - learning_rate: 0.0010
Epoch 3/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8004 - loss: 0.5213 - val_accuracy: 0.7267 - val_loss: 0.7605 - learning_rate: 0.0010
Epoch 4/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8455 - loss: 0.4150 - val_accuracy: 0.7851 - val_loss: 0.5756 - learning_rate: 0.0010
Epoch 5/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8678 - loss: 0.3493 - val_accuracy: 0.7820 - val_loss: 0.6971 - learning_rate: 0.0010
Epoch 6/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8862 - loss: 0.2983 - val_accuracy: 0.8874 - val_loss: 0.3212 - learning_rate: 0.0010
Epoch 7/50
173/173 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9071 - loss: 0.2532 

# Saved result So far

In [ ]:
day2_results = {
    'eegnet_subject_dependent': result_eegnet_sd,
    'eegnet_cross_subject': result_eegnet_cs,
    'cnn1d_subject_dependent': result_cnn_sd,
    'cnn1d_cross_subject': result_cnn_cs,
}

results_path = os.path.join(SAVE_DIR, 'day2_results.pkl')
with open(results_path, 'wb') as f:
    pickle.dump(day2_results, f)

print('saved day 2 results to', results_path)

for key, res in day2_results.items():
    print(key, '- accuracy:', round(res['accuracy'], 4), 'f1:', round(res['f1_score'], 4))

saved day 2 results to /content/drive/MyDrive/GAMEEMO_processed/results/day2_results.pkl
eegnet_subject_dependent - accuracy: 0.4412 f1: 0.438
eegnet_cross_subject - accuracy: 0.6075 f1: 0.6029
cnn1d_subject_dependent - accuracy: 0.7462 f1: 0.7497
cnn1d_cross_subject - accuracy: 0.9825 f1: 0.9825
